**descirption** In this section, feature engineering is performed to prepare data for the recommendation model. Both genre-based and tag-based representations are explored.

for:
one-hot encoding of genres  
feature-matris  
experiment with cosine similarity  
try KNN

# 1. Content-based Movie Recommender - genre

Features:
- Genres converted to text
- TF-IDF vectorization
- Cosine similarity recommendation

In [187]:
#Interactive library
import plotly.express as px

# Data handling
import numpy as np
import pandas as pd

# Machine learning
import sklearn

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.feature_extraction.text import TfidfVectorizer

# Similarity
from sklearn.metrics.pairwise import cosine_similarity as cosine_sim

# String matching
from rapidfuzz import process

import re


In [188]:
def normalize_title(title):
    title = title.lower()
    title = title.replace(" ", "")
    title = re.sub(r"\(\d{4}\)", "", title)
    return title

In [189]:
directory_data = "../data" # Adjust the path to your data directory if necessary
movies = pd.read_csv(f"{directory_data}/raw/movies.csv")  # Adjust the path to your data directory if necessary
ratings = pd.read_csv(f"{directory_data}/raw/ratings.csv")
tags = pd.read_csv(f"{directory_data}/raw/tags.csv")

### Content-based recommendation using TF-IDF and cosine similarity

This implementation was inspired by a discussion with Johan Challita on LinkedIn, and is used as a starting point for further exploration.

The initial idea was to begin with a simple content-based similarity recommender as a baseline. In this notebook, a TF-IDF based representation is used together with cosine similarity to identify similar movies.

Rather than implementing a separate KNN model, the nearest neighbours are retrieved directly by ranking movies based on cosine similarity. This approach is equivalent to a nearest neighbour search in the feature space and provides a simple and efficient baseline for recommendation.

### TF-IDF feature representation

To transform the movie data into a numerical representation, TF-IDF vectorization is applied to the genre information.

`TfidfVectorizer` defines how text should be converted into numerical features using the TF-IDF method.

At this stage, the vectorizer is only initialized with chosen settings, such as removing English stop words. It does not yet contain any information about the dataset.

In other words, this step defines *how* the text should be processed, but no transformation has been applied yet.

The genres are first converted into a space-separated text format. This allows them to be treated as tokens and makes it possible to apply standard text processing techniques such as TF-IDF.

---

### TF-IDF transformation (fit_transform)

The `fit_transform()` method performs two steps:

- **fit**: learns the vocabulary from the dataset and calculates how important each word is across all documents  
- **transform**: converts each document into a numerical vector based on these learned values  

The result is a TF-IDF matrix where:
- each row represents a movie  
- each column represents a word (feature)  
- each value represents how important that word is for that movie  

The matrix is stored as a *sparse matrix*, meaning that only non-zero values are stored. This makes it memory efficient, since most words do not appear in most movies.

English stop words are removed during vectorization. While the MovieLens dataset is not limited to English-language movies, this approach is still applicable since genres and many tags are expressed using English terms.

However, this simplification introduces some limitations, as language variations and inconsistencies in user-generated data are not fully addressed.

In [190]:
movies["content"] = movies["genres"].str.replace("|", " ", regex=False)


# Create TF-IDF matrix
tfidf = TfidfVectorizer(stop_words="english")   #for english stop words, you can adjust this for other languages or use a custom list
tfidf_matrix = tfidf.fit_transform(movies["content"])

In [191]:

def recommend_movies_by_genres(movie_title, n=5):
    matches = movies[movies["title"].str.contains(movie_title, case=False, na=False)]

    if len(matches) == 0:
        print(f"Movie could not be found")
        return None
    if len(matches) > 1:
        print("multiple movies matched your search: ")
        print(matches[["title"]].head(10))

    
    movie_index = matches.index[0]
    title = movies.loc[movie_index, "title"]

    similarity_score = cosine_sim(tfidf_matrix[movie_index], tfidf_matrix).flatten()
       

    similar_indexes = similarity_score.argsort()[::-1]
    similar_indexes = [i for i in similar_indexes if i != movie_index]

    result = movies.iloc[similar_indexes][["title", "genres"]].head(n).copy()

    print(f"Recommendations for '{title}':")
    print(f"Movies with similarity > 0: {(similarity_score > 0).sum() - 1}")
    print(f"Movies with similarity > 0.1: {(similarity_score > 0.1).sum() - 1}")
    print(f"Movies with similarity > 0.3: {(similarity_score > 0.3).sum() - 1}")
    print(f"Movies with similarity > 0.5: {(similarity_score > 0.5).sum() - 1}")


    return result

In [192]:
recommend_movies_by_genres("kung fu hustle", 5)

Recommendations for 'Kung Fu Hustle (Gong fu) (2004)':
Movies with similarity > 0: 30660
Movies with similarity > 0.1: 30660
Movies with similarity > 0.3: 25256
Movies with similarity > 0.5: 12866


,title,genres
64827,Take Home Pay (2019),Action|Comedy
70281,Cats & Dogs 3: Paws Unite (2020),Action|Comedy
4479,Disorganized Crime (1989),Action|Comedy
58972,Once Upon a Deadpool (2018),Action|Comedy
55743,Surge of Power: Revenge of the Sequel (2018),Action|Comedy


#Note to me: explore till KNN-solution! What differnece will it make?

The current model is based on TF-IDF representations and cosine similarity, which rely on word overlap rather than true semantic understanding. As a result, many movies receive relatively high similarity scores because they share common features (genres). This leads to a broad similarity structure where many movies are considered similar, even if they are not closely related in meaning.

# 2. Content-based Movie Recommender - genre and tags

Features:
- Genres converted to text
- TF-IDF vectorization
- combination tags and genre
- Cosine similarity recommendation

**tags per movie**

In [193]:
tags_per_movie = (
    tags.groupby("movieId")["tag"]
    .apply(lambda x: " ".join(x.dropna().astype(str)))
    .reset_index()
    .rename(columns={"tag": "tags_combined"})
)

In [194]:
print(movies.columns)

print(tags_per_movie["tags_combined"].isna().sum())

Index(['movieId', 'title', 'genres', 'content'], dtype='str')
0


**Merge tags into movies**

In [195]:
movies = movies.merge(tags_per_movie, on="movieId", how="left")



**fill missing tags**

In [196]:
movies["tags_combined"] = movies["tags_combined"].fillna("")

**Clean genres**

In [197]:
movies["genres_clean"] = movies["genres"].apply(
    lambda x: "" if x == "(no genres listed)" else x
)

**clean title**

In [198]:
movies["clean_title"] = movies["title"].apply(normalize_title)



**Build cobined test column**

*obs! Kolla genre_clean om det verkligen finns "-" eller "" i datan!*

In [199]:
movies["genres_clean"] = (
    movies["genres"]
    .str.replace("(no genres listed)", "", regex=False)
    .str.replace("|", " ", regex=False)
    .str.replace("-", "", regex=False)
    .str.lower()
)

movies["tags_clean"] = (
    movies["tags_combined"]
    .fillna("")
    .str.replace("-", "", regex=False)
    .str.lower()
)

movies["content_with_tags"] = (
    movies["genres_clean"]
    + " "
    + movies["tags_clean"].str.replace(r"\b\d+\b", 
    "", 
    regex=True
    )
)

movies["content_with_tags"] = (
    movies["content_with_tags"]
    .str.replace(r"[^a-zA-Z\s]", "", regex=True)   # Only keep latin letters and spaces
    .str.replace(r"\s+", " ", regex=True)          # Replace multiple spaces with a single space
    .str.strip()                                   # Remove leading and trailing spaces
)

**TF-IDF on combined test**

In [200]:
tfidf_tags = TfidfVectorizer(stop_words="english")
tfidf_matrix_tags = tfidf_tags.fit_transform(movies["content_with_tags"])
feature_names = tfidf_tags.get_feature_names_out()

print(feature_names[:20])
print(feature_names[1000:1020])
print(feature_names[-20:])

['aa' 'aaliyah' 'aaltonen' 'aames' 'aamir' 'aampm' 'aamuja' 'aanand'
 'aand' 'aang' 'aapi' 'aardman' 'aarn' 'aaron' 'aaryan' 'aashiqui' 'aasif'
 'aat' 'aayesha' 'aba']
['aliensmonsters' 'aligator' 'alighieri' 'alignment' 'alignments' 'alik'
 'alike' 'alimentation' 'alimony' 'alin' 'alina' 'aline' 'alins' 'alisa'
 'aliso' 'alison' 'alistair' 'alive' 'alize' 'aljazeera']
['zuzus' 'zver' 'zvyagintsev' 'zwanikken' 'zweig' 'zwelleger' 'zwerge'
 'zwerin' 'zweverig' 'zwick' 'zwigoff' 'zydeco' 'zyklon' 'zylberstein'
 'zyprexa' 'zyzek' 'zz' 'zzz' 'zzzax' 'zzzzzzzzzzzzzz']


**Handle different alphabets, signs and numbers**

A limitation of this approach is that it removes potentially meaningful non-English tags. However, given the scope of the model, this trade-off was made to improve consistency and interpretability.

Since the model uses English stop words and does not include language detection, non-Latin characters were removed from the dataset. This reduces noise from multilingual user-generated tags and ensures a more consistent feature space for similarity calculations.

In [201]:
feature_names = tfidf_tags.get_feature_names_out()

**Create new content**  
create new column and do not overwright the old ones.

In [213]:
tfidf_tags = TfidfVectorizer(stop_words="english")
tfidf_matrix_tags = tfidf_tags.fit_transform(movies["content_with_tags"])

feature_names = tfidf_tags.get_feature_names_out()

print(feature_names[:20])
print(feature_names[1000:1020])
print(feature_names[-20:])



['aa' 'aaliyah' 'aaltonen' 'aames' 'aamir' 'aampm' 'aamuja' 'aanand'
 'aand' 'aang' 'aapi' 'aardman' 'aarn' 'aaron' 'aaryan' 'aashiqui' 'aasif'
 'aat' 'aayesha' 'aba']
['aliensmonsters' 'aligator' 'alighieri' 'alignment' 'alignments' 'alik'
 'alike' 'alimentation' 'alimony' 'alin' 'alina' 'aline' 'alins' 'alisa'
 'aliso' 'alison' 'alistair' 'alive' 'alize' 'aljazeera']
['zuzus' 'zver' 'zvyagintsev' 'zwanikken' 'zweig' 'zwelleger' 'zwerge'
 'zwerin' 'zweverig' 'zwick' 'zwigoff' 'zydeco' 'zyklon' 'zylberstein'
 'zyprexa' 'zyzek' 'zz' 'zzz' 'zzzax' 'zzzzzzzzzzzzzz']


**Function för genre + tags**

In [214]:
def recommend_movies_by_genres_and_tags(movie_title, n=5):
    matches = movies[movies["title"].str.contains(movie_title, case=False, na=False)]

    if len(matches) == 0:
        print("Movie could not be found")
        return None
    
    if len(matches) > 1:
        print("Multiple movies found. Please be more specific: ")
        print(matches[["title"]].head(10))

    movie_index = matches.index[0]
    title = movies.loc[movie_index, "title"]

    similarity_score = cosine_sim(
        tfidf_matrix_tags[movie_index],
        tfidf_matrix_tags
    ).flatten()

    similar_indexes = similarity_score.argsort()[::-1]
    similar_indexes = [i for i in similar_indexes if i != movie_index]

    result = movies.iloc[similar_indexes][["title", "genres", "tags_combined"]].head(n).copy()

    return result

    

**test function**

In [204]:
recommend_movies_by_genres_and_tags("Toy Story", 5)


Multiple movies found. Please be more specific: 
                                           title
0                               Toy Story (1995)
3021                          Toy Story 2 (1999)
14815                         Toy Story 3 (2010)
20507                 Toy Story of Terror (2013)
22654  Toy Story Toons: Hawaiian Vacation (2011)
22655          Toy Story Toons: Small Fry (2011)
24101    Toy Story Toons: Partysaurus Rex (2012)
24103          Toy Story That Time Forgot (2014)
60793                         Toy Story 4 (2019)
Recommendations for 'Toy Story (1995)' based on genres and tags:
Movies with similarity < 0: 56023
Movies with similarity < 0.1: 3877
Movies with similarity < 0.3: 906
Movies with similarity < 0.5: 33


,title,genres,tags_combined
3021,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,whimsical cgi Disney Pixar very good but the f...
2264,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,animation Disney Pixar acting circus coming of...
14815,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,Pixar Pixar Annemari dolls escape good versus ...
39883,Finding Dory (2016),Adventure|Animation|Comedy,adventure amnesia Animation boring dumb famil...
4781,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,cute funny innovative Disney Pixar cute funny ...


**check which words dictate the recommendation?**

**Choose a movie**

In [218]:
movie_title = "Toy Story"
matches = movies[movies["title"].str.contains(movie_title, case=False, na=False)]


In [219]:
print(matches[["title"]].head())

print(movies[movies["title"] == movie_title])

                                           title
0                               Toy Story (1995)
3021                          Toy Story 2 (1999)
14815                         Toy Story 3 (2010)
20507                 Toy Story of Terror (2013)
22654  Toy Story Toons: Hawaiian Vacation (2011)
Empty DataFrame
Columns: [movieId, title, genres, content, tags_combined, genres_clean, clean_title, tags_clean, content_with_tags]
Index: []


In [221]:
print(movies[movies["title"].str.contains("Toy", case=False, na=False)][["title"]].head())


                                      title
0                          Toy Story (1995)
1928                Babes in Toyland (1961)
2162                            Toys (1992)
2389  Dry Cleaning (Nettoyage à sec) (1997)
2993                Babes in Toyland (1934)


**Find index**

In [224]:
movie_index = matches.index[0]

print(movie_index)

0
